# Exercício 11 — Lista duplamente encadeada e Deque

**Henrique Goldstein Maluhy Mendes Amaral** · DR1_AT — Algoritmos e Estruturas de Dados · Infnet

Para rodar: **Ambiente de execução → Executar tudo**. O notebook é independente, não precisa de nenhum arquivo externo.

**Roteiro:** (1) `DoublyLinkedList` com `prev` e `next` · (2) `Deque` construído sobre ela · (3) as invariantes e como verificar · (4) bateria de testes alternando as duas pontas · (5) evidência de O(1) · (6) casos de borda

In [1]:
import random
import time

import pandas as pd

random.seed(42)

## 1. A lista duplamente encadeada

Cada nó aponta para o **próximo** (`next`) e para o **anterior** (`prev`). É esse `prev` que falta na lista simples: com ele, remover o último nó é O(1), porque do último eu chego no penúltimo sem percorrer a lista.

**Decisões:**
- **Estrutura vazia** = `head` e `tail` iguais a `None` e `size` igual a 0. Remover de uma estrutura vazia levanta `EmptyDequeError` com mensagem clara. Não devolvo `None`, porque `None` pode ser um valor guardado de verdade.
- **Gerenciamento de nós:** quando um nó sai, zero o `prev` e o `next` dele. Assim ele não fica segurando referência para nós que continuam na lista, e se alguém guardou esse nó por engano, ele não dá acesso à estrutura.

In [2]:
class EmptyDequeError(Exception):
    """Remoção ou consulta numa estrutura vazia."""


class Node:
    def __init__(self, value):
        self.value = value
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def is_empty(self):
        return self.size == 0

    def __len__(self):
        return self.size

    def insert_first(self, value):
        novo = Node(value)
        if self.head is None:
            self.head = self.tail = novo           # único nó: é o primeiro e o último
        else:
            novo.next = self.head
            self.head.prev = novo
            self.head = novo
        self.size += 1

    def insert_last(self, value):
        novo = Node(value)
        if self.tail is None:
            self.head = self.tail = novo
        else:
            novo.prev = self.tail
            self.tail.next = novo
            self.tail = novo
        self.size += 1

    def delete_first(self):
        if self.head is None:
            raise EmptyDequeError("delete_first: a lista está vazia")
        no = self.head
        self.head = no.next
        if self.head is None:
            self.tail = None                       # era o último nó: a lista esvaziou
        else:
            self.head.prev = None
        no.next = None                             # o nó removido não aponta mais para a lista
        self.size -= 1
        return no.value

    def delete_last(self):
        if self.tail is None:
            raise EmptyDequeError("delete_last: a lista está vazia")
        no = self.tail
        self.tail = no.prev                        # o prev dá o penúltimo direto: O(1)
        if self.tail is None:
            self.head = None
        else:
            self.tail.next = None
        no.prev = None
        self.size -= 1
        return no.value

    def to_list(self):
        valores, atual = [], self.head
        while atual is not None:
            valores.append(atual.value)
            atual = atual.next
        return valores

    def to_list_reverse(self):
        valores, atual = [], self.tail
        while atual is not None:
            valores.append(atual.value)
            atual = atual.prev
        return valores

## 2. O `Deque` sobre a lista

O deque **usa** uma lista duplamente encadeada por dentro (composição), em vez de herdar dela. Assim a interface do deque tem **só** as operações das pontas, todas O(1): quem usa o deque não consegue chamar, por acidente, nada que percorra a lista.

In [3]:
class Deque:
    def __init__(self):
        self._lista = DoublyLinkedList()

    def is_empty(self):
        return self._lista.is_empty()

    def __len__(self):
        return len(self._lista)

    def insert_left(self, value):
        self._lista.insert_first(value)

    def insert_right(self, value):
        self._lista.insert_last(value)

    def remove_left(self):
        if self.is_empty():
            raise EmptyDequeError("remove_left: o deque está vazio")
        return self._lista.delete_first()

    def remove_right(self):
        if self.is_empty():
            raise EmptyDequeError("remove_right: o deque está vazio")
        return self._lista.delete_last()

    def peek_left(self):
        if self.is_empty():
            raise EmptyDequeError("peek_left: o deque está vazio")
        return self._lista.head.value

    def peek_right(self):
        if self.is_empty():
            raise EmptyDequeError("peek_right: o deque está vazio")
        return self._lista.tail.value

    def to_list(self):
        return self._lista.to_list()


d = Deque()
d.insert_right(2)
d.insert_right(3)
d.insert_left(1)
print("deque:", d.to_list(), "| esquerda:", d.peek_left(), "| direita:", d.peek_right())
print("remove_left:", d.remove_left(), "| remove_right:", d.remove_right(), "| sobra:", d.to_list())

deque: [1, 2, 3] | esquerda: 1 | direita: 3
remove_left: 1 | remove_right: 3 | sobra: [2]


## 3. As invariantes

Depois de qualquer operação, a lista precisa satisfazer:

1. `size` é igual ao número de nós percorridos do `head` até o fim;
2. `head.prev` é `None` e `tail.next` é `None`;
3. para todo nó, `no.next.prev` é o próprio nó (os dois ponteiros concordam);
4. percorrer **de trás para frente** dá exatamente a sequência de ida **invertida**;
5. vazia ⇔ `head is None` ⇔ `tail is None`.

A 3ª e a 4ª são as que pegam os erros típicos de lista dupla: esquecer de atualizar um dos dois ponteiros deixa a ida certa e a volta errada.

In [4]:
def verificar(lista):
    if lista.size == 0:
        assert lista.head is None and lista.tail is None, "lista vazia com head/tail sobrando"
        return
    assert lista.head.prev is None, "head.prev deveria ser None"
    assert lista.tail.next is None, "tail.next deveria ser None"
    contados, atual = 0, lista.head
    while atual is not None:
        contados += 1
        assert contados <= lista.size, "mais nós do que o size (ciclo?)"
        if atual.next is not None:
            assert atual.next.prev is atual, f"ponteiros discordam no nó {atual.value!r}"
        atual = atual.next
    assert contados == lista.size, f"size = {lista.size}, mas há {contados} nós"
    assert lista.to_list_reverse() == lista.to_list()[::-1], "a volta não é a ida invertida"


# a verificação precisa FALHAR quando a estrutura está errada, senão ela não prova nada
quebrada = DoublyLinkedList()
for v in [1, 2, 3]:
    quebrada.insert_last(v)
quebrada.head.next.prev = None          # estrago um ponteiro de propósito
try:
    verificar(quebrada)
    print("ERRO: a verificação deixou passar uma lista corrompida")
except AssertionError as erro:
    print(f"lista corrompida de propósito → detectado: {erro}")

lista corrompida de propósito → detectado: ponteiros discordam no nó 1


## 4. Bateria de testes alternando as duas pontas

Cada sequência roda num deque novo, com uma lista do Python fazendo as mesmas operações como **gabarito**. Depois de **cada operação** confiro o conteúdo e as invariantes.

In [5]:
def executar(nome, operacoes):
    deque, gabarito = Deque(), []
    for op, valor in operacoes:
        if op == "insert_left":
            deque.insert_left(valor)
            gabarito.insert(0, valor)
        elif op == "insert_right":
            deque.insert_right(valor)
            gabarito.append(valor)
        elif op == "remove_left":
            assert deque.remove_left() == gabarito.pop(0)
        elif op == "remove_right":
            assert deque.remove_right() == gabarito.pop()
        verificar(deque._lista)
        assert deque.to_list() == gabarito
        if gabarito:
            assert deque.peek_left() == gabarito[0] and deque.peek_right() == gabarito[-1]
    return {"sequência": nome, "operações": len(operacoes), "tamanho final": len(deque),
            "invariantes": "ok em todos os passos"}


n = 500
sequencias = {
    "entra à esquerda, sai à direita (fila)":
        [("insert_left", i) for i in range(n)] + [("remove_right", None)] * n,
    "entra e sai à direita (pilha)":
        [("insert_right", i) for i in range(n)] + [("remove_right", None)] * n,
    "entra alternando, sai alternando":
        [("insert_left" if i % 2 else "insert_right", i) for i in range(n)]
        + [("remove_left" if i % 2 else "remove_right", None) for i in range(n)],
    "enche e esvazia várias vezes":
        ([("insert_right", i) for i in range(50)] + [("remove_left", None)] * 50) * 10,
}

# e uma sequência longa sorteada, que nunca tenta remover de deque vazio
aleatoria, tamanho = [], 0
for i in range(20_000):
    if tamanho == 0 or random.random() < 0.5:
        aleatoria.append((random.choice(["insert_left", "insert_right"]), i))
        tamanho += 1
    else:
        aleatoria.append((random.choice(["remove_left", "remove_right"]), None))
        tamanho -= 1
sequencias["20.000 operações sorteadas nas duas pontas"] = aleatoria

pd.DataFrame([executar(nome, ops) for nome, ops in sequencias.items()])

,sequência,operações,tamanho final,invariantes
0,"entra à esquerda, sai à direita (fila)",1000,0,ok em todos os passos
1,entra e sai à direita (pilha),1000,0,ok em todos os passos
2,"entra alternando, sai alternando",1000,0,ok em todos os passos
3,enche e esvazia várias vezes,1000,0,ok em todos os passos
4,20.000 operações sorteadas nas duas pontas,20000,290,ok em todos os passos


As cinco sequências passaram com as invariantes conferidas **a cada operação** — 24 mil verificações no total. As três primeiras testam os usos clássicos (fila, pilha e as duas pontas intercaladas); a quarta testa a passagem repetida pelo estado vazio, que é onde `head` e `tail` precisam voltar a `None` juntos.

## 5. Evidência de O(1)

Se as operações são O(1), o tempo **por operação** não pode depender do tamanho do deque. Encho deques de tamanhos bem diferentes e meço 20.000 operações misturadas em cada um. Para comparar, faço o mesmo com uma **lista do Python usada como deque**, onde tirar e pôr na esquerda (`insert(0, x)` e `pop(0)`) desloca todos os elementos:

In [6]:
def medir(tamanho):
    deque, lista = Deque(), []
    for i in range(tamanho):
        deque.insert_right(i)
        lista.append(i)
    ops = [random.choice(["esq", "dir"]) for _ in range(20_000)]

    inicio = time.perf_counter()
    for op in ops:                           # entra e sai do mesmo lado: o tamanho não muda
        if op == "esq":
            deque.insert_left(0)
            deque.remove_left()
        else:
            deque.insert_right(0)
            deque.remove_right()
    tempo_deque = time.perf_counter() - inicio

    inicio = time.perf_counter()
    for op in ops:
        if op == "esq":
            lista.insert(0, 0)
            lista.pop(0)
        else:
            lista.append(0)
            lista.pop()
    tempo_lista = time.perf_counter() - inicio
    return tempo_deque, tempo_lista


linhas = []
for tamanho in [1_000, 10_000, 100_000, 400_000]:
    tempo_deque, tempo_lista = medir(tamanho)
    linhas.append({"tamanho do deque": tamanho,
                   "µs por operação — Deque": tempo_deque / 40_000 * 1e6,
                   "µs por operação — lista do Python": tempo_lista / 40_000 * 1e6})

o1 = pd.DataFrame(linhas)
o1["Deque: variação vs menor tamanho"] = o1["µs por operação — Deque"] / o1["µs por operação — Deque"].iloc[0]
o1

,tamanho do deque,µs por operação — Deque,µs por operação — lista do Python,Deque: variação vs menor tamanho
0,1000,0.176087,0.085377,1.000000
1,10000,0.220717,5.261830,1.253454
2,100000,0.203943,52.917735,1.158188
3,400000,0.159373,129.021150,0.905076


- **Deque:** o tempo por operação fica praticamente o mesmo com o deque 400 vezes maior — é O(1). Nenhuma operação percorre a lista: todas mexem só em `head` ou `tail` e em seus vizinhos imediatos.
- **Lista do Python:** o lado esquerdo é O(n), porque `insert(0, x)` e `pop(0)` deslocam todo o vetor. O tempo por operação cresce com o tamanho.
- **Com 1.000 elementos a lista do Python ainda ganha:** deslocar mil referências num bloco contínuo de memória é muito rápido, mais rápido que criar um objeto `Node` e religar ponteiros. A vantagem do Deque não é ser rápido em qualquer tamanho — é **não depender do tamanho**. A partir de alguns milhares de elementos, a lista perde, e a diferença só aumenta.

### Custo por operação

| operação | custo | por quê |
|---|---|---|
| `insert_left` / `insert_first` | O(1) | liga o nó novo antes do `head` |
| `insert_right` / `insert_last` | O(1) | liga o nó novo depois do `tail` |
| `remove_left` / `delete_first` | O(1) | o `head` avança um nó |
| `remove_right` / `delete_last` | **O(1)** | o `tail.prev` dá o penúltimo direto (na lista simples era O(n)) |
| `peek_left` / `peek_right` | O(1) | leitura de `head` / `tail` |
| `is_empty` / `__len__` | O(1) | `size` mantido a cada operação |
| verificar invariantes | O(n) | percorre ida e volta — usado só nos testes |

O preço do O(1) nas duas pontas é **um ponteiro a mais por nó** (o `prev`) e o cuidado de atualizar os dois ponteiros em toda operação — que é exatamente o que a bateria de testes confere.

## 6. Casos de borda

In [7]:
vazio = Deque()
assert vazio.is_empty() and len(vazio) == 0

# as quatro consultas/remoções em deque vazio dão erro claro, e o deque continua utilizável
for operacao in (vazio.remove_left, vazio.remove_right, vazio.peek_left, vazio.peek_right):
    try:
        operacao()
        raise AssertionError("deveria ter levantado EmptyDequeError")
    except EmptyDequeError as erro:
        print(f"{erro}")
verificar(vazio._lista)

# um elemento só: ele é a esquerda e a direita ao mesmo tempo
vazio.insert_left("x")
assert vazio.peek_left() == vazio.peek_right() == "x"
assert vazio._lista.head is vazio._lista.tail
assert vazio.remove_right() == "x" and vazio.is_empty()
verificar(vazio._lista)

# None é um valor válido e não se confunde com "vazio"
vazio.insert_right(None)
assert len(vazio) == 1 and vazio.peek_left() is None and vazio.remove_left() is None

# o nó removido não fica apontando para a lista
lista = DoublyLinkedList()
for v in [1, 2, 3]:
    lista.insert_last(v)
primeiro = lista.head
lista.delete_first()
assert primeiro.next is None and primeiro.prev is None

print("\ntodos os casos de borda passaram")

remove_left: o deque está vazio
remove_right: o deque está vazio
peek_left: o deque está vazio
peek_right: o deque está vazio

todos os casos de borda passaram
